In [0]:
%run "./00_Pipeline_Configurations"

In [0]:
%run "./08_Audit_Log_Function"

In [0]:
from datetime import datetime

start_time = datetime.now()

try:
    df_students_gold = spark.table(gold_students_table)
    df_attendance_gold = spark.table(gold_attendance_table)

    df_final = df_students_gold.join(df_attendance_gold, on="student_id", how="inner")

    df_final.write.format("delta").mode("overwrite").saveAsTable(master_gold_table)

    display(df_final)

    print(f"Total rows in {master_gold_table}: {df_final.count()}")

    log_audit("gold_final_join", table_name=master_gold_table, status="SUCCESS", start_time=start_time)

except Exception as e:
    log_audit("gold_final_join", status="FAILED", start_time=start_time, error_msg=str(e))
    raise